# Experiment version one — data validation

This notebook validates the saved Liu2024 dataset and the three evaluation protocols before model development. No model is trained here.

In [1]:
import sys
from pathlib import Path

def find_repository_root(start=Path.cwd()):
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / 'src' / 'ourexperimentversionone').is_dir():
            return directory
    raise FileNotFoundError('Could not locate the EEG repository root')

REPOSITORY_ROOT = find_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
print('Repository:', REPOSITORY_ROOT)

Repository: /workspaces/EEG


## 1. Load the saved dataset

In [2]:
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import RandomSampler, SequentialSampler

from src.datautils.MoabbLiu2024 import Liu2024TorchDataset
from src.ourexperimentversionone.data import (
    DataLoaderConfig,
    create_group_kfold_dataloaders,
    create_loso_dataloaders,
    create_within_subject_dataloaders,
    group_kfold_splits,
    loso_split,
    within_subject_splits,
)

dataset = Liu2024TorchDataset()
loader_config = DataLoaderConfig(batch_size=32, pin_memory=False, seed=42)
label_names = {value: key for key, value in dataset.metadata['class_mapping'].items()}

print('Directory:', dataset.data_dir)
print('X:', dataset.X.shape, dataset.X.dtype)
print('y:', dataset.y.shape, dataset.y.dtype)
print('Subjects:', len(np.unique(dataset.subject_ids)))
print('Class counts:', dict(sorted(Counter(map(int, dataset.y)).items())))
print('Metadata preprocessing:', dataset.metadata['preprocessing'])

Directory: /workspaces/EEG/data/moabb/Preprocessed-MNE-liu2024-data
X: (2000, 29, 2000) float32
y: (2000,) int64
Subjects: 50
Class counts: {0: 1000, 1: 1000}
Metadata preprocessing: {'bandpass_hz': [8.0, 32.0], 'standardize': True, 'standardize_factor_new': 0.001, 'standardize_init_block_seconds': 4.0, 'standardize_init_block_size': 2000, 'trial_start_offset_seconds': 0.0, 'trial_stop_offset_seconds': 0.0}


## 2. Dataset integrity checks

In [3]:
subjects = np.unique(dataset.subject_ids).astype(int)
class_counts = Counter(map(int, dataset.y))
subject_trial_counts = Counter(map(int, dataset.subject_ids))

assert dataset.X.shape == (2000, 29, 2000)
assert dataset.X.dtype == np.float32
assert set(map(int, dataset.y)) == {0, 1}
assert class_counts == Counter({0: 1000, 1: 1000})
assert np.array_equal(subjects, np.arange(1, 51))
assert set(subject_trial_counts.values()) == {40}
assert np.isfinite(dataset.X).all()
assert dataset.metadata['window_shape'] == [29, 2000]
assert dataset.metadata['sampling_frequency_hz'] == 500.0
print('All saved-dataset integrity checks passed.')

All saved-dataset integrity checks passed.


## 3. Five-fold grouped cross-validation

Subjects—not individual trials—are separated across training, validation, and test partitions.

In [4]:
group_folds = group_kfold_splits(dataset.subject_ids, seed=loader_config.seed)
group_rows = []
all_test_subjects = []
for split in group_folds:
    train_subjects = set(split.train_subject_ids)
    validation_subjects = set(split.validation_subject_ids)
    test_subjects = set(split.test_subject_ids)
    assert train_subjects.isdisjoint(validation_subjects)
    assert train_subjects.isdisjoint(test_subjects)
    assert validation_subjects.isdisjoint(test_subjects)
    assert len(split.train_indices) == 1280
    assert len(split.validation_indices) == 320
    assert len(split.test_indices) == 400
    assert Counter(map(int, dataset.y[split.train_indices])) == Counter({0: 640, 1: 640})
    assert Counter(map(int, dataset.y[split.validation_indices])) == Counter({0: 160, 1: 160})
    assert Counter(map(int, dataset.y[split.test_indices])) == Counter({0: 200, 1: 200})
    all_test_subjects.extend(split.test_subject_ids)
    group_rows.append({
        'fold': split.fold,
        'train_subjects': len(split.train_subject_ids),
        'validation_subjects': len(split.validation_subject_ids),
        'test_subjects': len(split.test_subject_ids),
        'train_trials': len(split.train_indices),
        'validation_trials': len(split.validation_indices),
        'test_trials': len(split.test_indices),
        'test_subject_ids': split.test_subject_ids,
    })

assert Counter(all_test_subjects) == Counter(range(1, 51))
group_summary = pd.DataFrame(group_rows)
display(group_summary)
print('All GroupKFold checks passed; every subject is tested exactly once.')

,fold,train_subjects,validation_subjects,test_subjects,train_trials,validation_trials,test_trials,test_subject_ids
0,0,32,8,10,1280,320,400,"(6, 18, 21, 26, 28, 40, 46, 47, 48, 50)"
1,1,32,8,10,1280,320,400,"(5, 8, 19, 24, 25, 30, 32, 38, 39, 41)"
2,2,32,8,10,1280,320,400,"(10, 16, 17, 22, 27, 29, 33, 35, 42, 43)"
3,3,32,8,10,1280,320,400,"(4, 7, 11, 12, 20, 23, 31, 36, 44, 45)"
4,4,32,8,10,1280,320,400,"(1, 2, 3, 9, 13, 14, 15, 34, 37, 49)"


All GroupKFold checks passed; every subject is tested exactly once.


In [5]:
group_loaders = create_group_kfold_dataloaders(fold=0, config=loader_config, dataset=dataset)
x, y, info = next(iter(group_loaders.train))

assert x.shape == (32, 29, 2000)
assert x.dtype == torch.float32
assert y.shape == (32,)
assert set(info) == {'subject', 'trial'}
assert isinstance(group_loaders.train.sampler, RandomSampler)
assert isinstance(group_loaders.validation.sampler, SequentialSampler)
assert isinstance(group_loaders.test.sampler, SequentialSampler)
assert set(map(int, info['subject'].tolist())).issubset(set(group_loaders.split.train_subject_ids))

print('Training batch:', x.shape, x.dtype)
print('Batch labels:', Counter(map(int, y.tolist())))
print('Batch subjects:', sorted(set(map(int, info['subject'].tolist()))))
print('GroupKFold DataLoader checks passed.')

Training batch: torch.Size([32, 29, 2000]) torch.float32
Batch labels: Counter({1: 17, 0: 15})
Batch subjects: [1, 2, 8, 9, 10, 11, 13, 14, 15, 16, 19, 24, 25, 27, 30, 38, 39, 42, 43, 44, 45]
GroupKFold DataLoader checks passed.


## 4. Leave-one-subject-out evaluation

In [6]:
loso_rows = []
for test_subject in range(1, 51):
    split = loso_split(dataset.subject_ids, test_subject_id=test_subject)
    assert len(split.train_subject_ids) == 49
    assert len(split.validation_subject_ids) == 0
    assert split.test_subject_ids == (test_subject,)
    assert len(split.train_indices) == 1960
    assert len(split.validation_indices) == 0
    assert len(split.test_indices) == 40
    assert Counter(map(int, dataset.y[split.test_indices])) == Counter({0: 20, 1: 20})
    loso_rows.append({
        'test_subject': test_subject,
        'training_subjects': len(split.train_subject_ids),
    })

display(pd.DataFrame(loso_rows).head(10))
print('All 50 LOSO folds passed.')

,test_subject,training_subjects
0,1,49
1,2,49
2,3,49
3,4,49
4,5,49
5,6,49
6,7,49
7,8,49
8,9,49
9,10,49


All 50 LOSO folds passed.


In [7]:
loso_loaders = create_loso_dataloaders(test_subject_id=1, config=loader_config, dataset=dataset)
test_x, test_y, test_info = next(iter(loso_loaders.test))
assert test_x.shape == (32, 29, 2000)
assert set(map(int, test_info['subject'].tolist())) == {1}
assert loso_loaders.validation is None
assert len(loso_loaders.split.train_subject_ids) == 49
print('LOSO test batch:', test_x.shape)
print('Test subject:', loso_loaders.split.test_subject_ids)
print('Training subjects:', len(loso_loaders.split.train_subject_ids))

LOSO test batch: torch.Size([32, 29, 2000])
Test subject: (1,)
Training subjects: 49


## 5. Within-subject evaluation

In [8]:
SUBJECT_ID = 1
within_folds = within_subject_splits(
    dataset.subject_ids, dataset.y, subject_id=SUBJECT_ID, seed=loader_config.seed
)
for split in within_folds:
    assert (len(split.train_indices), len(split.validation_indices), len(split.test_indices)) == (24, 8, 8)
    assert Counter(map(int, dataset.y[split.train_indices])) == Counter({0: 12, 1: 12})
    assert Counter(map(int, dataset.y[split.validation_indices])) == Counter({0: 4, 1: 4})
    assert Counter(map(int, dataset.y[split.test_indices])) == Counter({0: 4, 1: 4})
    combined = np.concatenate([split.train_indices, split.validation_indices, split.test_indices])
    assert len(np.unique(combined)) == 40
    assert set(map(int, dataset.subject_ids[combined])) == {SUBJECT_ID}

within_loaders = create_within_subject_dataloaders(
    subject_id=SUBJECT_ID, fold=0, config=loader_config, dataset=dataset
)
within_x, within_y, within_info = next(iter(within_loaders.train))
assert within_x.shape == (24, 29, 2000)
assert set(map(int, within_info['subject'].tolist())) == {SUBJECT_ID}
print('All five within-subject folds passed.')
print('Training batch:', within_x.shape)

All five within-subject folds passed.
Training batch: torch.Size([24, 29, 2000])


## 6. Reproducibility and final summary

In [9]:
repeated_folds = group_kfold_splits(dataset.subject_ids, seed=loader_config.seed)
for original, repeated in zip(group_folds, repeated_folds):
    assert original.train_subject_ids == repeated.train_subject_ids
    assert original.validation_subject_ids == repeated.validation_subject_ids
    assert original.test_subject_ids == repeated.test_subject_ids
    assert np.array_equal(original.train_indices, repeated.train_indices)

different_seed_folds = group_kfold_splits(dataset.subject_ids, seed=loader_config.seed + 1)
assert any(
    first.test_subject_ids != second.test_subject_ids
    for first, second in zip(group_folds, different_seed_folds)
)

print('Reproducibility checks passed.')
print('DATA PIPELINE READY: integrity, balance, leakage, folds, batches, and metadata verified.')

Reproducibility checks passed.
DATA PIPELINE READY: integrity, balance, leakage, folds, batches, and metadata verified.


# Model Experiment

# Training Experiment